# Film Recommendation Engine

A content-based movie recommendation system built on the **TMDb 5000 dataset**.
Give it a movie you love, and it returns 5 films you'll probably love too — by matching directors, actors, plot keywords, and genres through nearest-neighbor search, then ranking candidates by popularity and release proximity.

**Three strategies combined:**
- **Content-Based** — Binary feature matrix from director, cast, keywords, and genres; 31 nearest neighbors via Euclidean distance
- **Popularity-Weighted** — Scores using `IMDB² × φ(votes) × φ(year)` Gaussian weighting
- **Sequel Detection** — Fuzzy string matching to deduplicate franchise entries

## Setup

**Required packages:** numpy, pandas, matplotlib, seaborn, scikit-learn, nltk, thefuzz, python-Levenshtein, wordcloud

```bash
pip install numpy pandas scikit-learn nltk thefuzz python-Levenshtein matplotlib seaborn wordcloud
```

**Dataset:** [TMDb 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata) — place `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` in a `dataset/` directory.

In [1]:
import os
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import wordnet
from sklearn.neighbors import NearestNeighbors
from thefuzz import fuzz
from wordcloud import WordCloud

plt.rcParams['patch.force_edgecolor'] = True
plt.style.use('fivethirtyeight')
mpl.rc('patch', edgecolor='dimgray', linewidth=1)
pd.options.display.max_columns = 50
%matplotlib inline
warnings.filterwarnings('ignore')

PS = nltk.stem.PorterStemmer()

DATA_DIR = 'dataset'
print(os.listdir(DATA_DIR))

LOST_COLUMNS = [
    'actor_1_facebook_likes', 'actor_2_facebook_likes', 'actor_3_facebook_likes',
    'aspect_ratio', 'cast_total_facebook_likes', 'color', 'content_rating',
    'director_facebook_likes', 'facenumber_in_poster', 'movie_facebook_likes',
    'movie_imdb_link', 'num_critic_for_reviews', 'num_user_for_reviews',
]

TMDB_TO_IMDB = {
    'budget': 'budget', 'genres': 'genres', 'revenue': 'gross',
    'title': 'movie_title', 'runtime': 'duration',
    'original_language': 'language', 'keywords': 'plot_keywords',
    'vote_count': 'num_voted_users',
}

IMDB_COLUMNS_TO_REMAP = {'imdb_score': 'vote_average'}


def load_tmdb_movies(path):
    """Load and parse the TMDb movies CSV."""
    df = pd.read_csv(path)
    df['release_date'] = pd.to_datetime(df['release_date']).apply(lambda x: x.date())
    for col in ['genres', 'keywords', 'production_countries', 'production_companies', 'spoken_languages']:
        df[col] = df[col].apply(json.loads)
    return df


def load_tmdb_credits(path):
    """Load and parse the TMDb credits CSV."""
    df = pd.read_csv(path)
    for col in ['cast', 'crew']:
        df[col] = df[col].apply(json.loads)
    return df

In [2]:
def safe_access(container, index_values):
    """Safely traverse nested containers, returning NaN on failure."""
    result = container
    try:
        for i in index_values:
            result = result[i]
        return result
    except (IndexError, KeyError):
        return np.nan


def pipe_flatten_names(keywords):
    return '|'.join([x['name'] for x in keywords])


def get_director(crew_data):
    directors = [x['name'] for x in crew_data if x['job'] == 'Director']
    return safe_access(directors, [0])


def convert_to_original_format(movies, credits):
    """Reshape TMDb data into the IMDb-style column layout."""
    df = movies.copy()
    df.rename(columns=TMDB_TO_IMDB, inplace=True)
    df['title_year'] = pd.to_datetime(df['release_date']).apply(lambda x: x.year)
    df['country'] = df['production_countries'].apply(lambda x: safe_access(x, [0, 'name']))
    df['language'] = df['spoken_languages'].apply(lambda x: safe_access(x, [0, 'name']))
    df['genres'] = df['genres'].apply(pipe_flatten_names)
    df['plot_keywords'] = df['plot_keywords'].apply(pipe_flatten_names)
    df['director_name'] = credits['crew'].apply(get_director)
    df['actor_1_name'] = credits['cast'].apply(lambda x: safe_access(x, [1, 'name']))
    df['actor_2_name'] = credits['cast'].apply(lambda x: safe_access(x, [2, 'name']))
    df['actor_3_name'] = credits['cast'].apply(lambda x: safe_access(x, [3, 'name']))
    return df

In [3]:
credits = load_tmdb_credits(os.path.join(DATA_DIR, 'tmdb_5000_credits.csv'))
movies = load_tmdb_movies(os.path.join(DATA_DIR, 'tmdb_5000_movies.csv'))
df_initial = convert_to_original_format(movies, credits)

# Summary of column types and null values
df_info = pd.DataFrame(df_initial.dtypes).T.rename(index={0: 'column type'})
df_info = pd.concat([df_info, pd.DataFrame(df_initial.isnull().sum()).T.rename(index={0: 'null values'})])
df_info = pd.concat([df_info, pd.DataFrame(df_initial.isnull().sum() / df_initial.shape[0] * 100).T.rename(index={0: 'null values (%)'})])
df_info

In order to build a recommendation engine, we would have to use keywords that describe the movies. All the keywords can be collected and we build models that work on the assumption that similar keywords should have similar contents.

In [4]:
set_keywords = set()
for list_keywords in df_initial['plot_keywords'].str.split('|').values:
    if isinstance(list_keywords, float):
        continue
    set_keywords = set_keywords.union(list_keywords)
set_keywords.discard('')

In [5]:
def count_word(df, ref_col, word_list):
    """Count occurrences of each word in the pipe-delimited column."""
    keyword_count = {s: 0 for s in word_list}
    for list_of_keywords in df[ref_col].str.split('|'):
        if isinstance(list_of_keywords, float) and pd.isnull(list_of_keywords):
            continue
        for s in list_of_keywords:
            if s in keyword_count and pd.notnull(s):
                keyword_count[s] += 1
    keyword_occurrences = sorted(keyword_count.items(), key=lambda x: x[1], reverse=True)
    return keyword_occurrences, keyword_count

In [6]:
keyword_occurrences, count = count_word(df_initial, 'plot_keywords', set_keywords)
keyword_occurrences[:10]

Representing the following top 500 keywords from the list of keywords using wordcloud, where the size of the word is directly proportional to its frequency.

In [7]:
fig = plt.figure(1, figsize=(18, 13))
ax = fig.add_subplot(2, 1, 1)

words_dict = {s[0]: s[1] for s in keyword_occurrences[:500]}

wc = WordCloud(
    width=1000, height=300,
    background_color='black',
    max_words=500,
    relative_scaling=1,
    normalize_plurals=False,
)
wc.generate_from_frequencies(words_dict)
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')

Representing the following top 50 keywords from the list of keywords using a histogram, where the X-axis represents the keywords and the Y-axis represents the number of occurences.

In [8]:
sub = keyword_occurrences[:50]
fig = plt.figure(1, figsize=(18, 13))
ax = fig.add_subplot(2, 1, 1)
y_vals = [i[1] for i in sub]
x_vals = list(range(len(sub)))
x_labels = [i[0] for i in sub]
plt.xticks(x_vals, x_labels, rotation=85, fontsize=15)
plt.yticks(fontsize=15)
plt.ylabel('Number of Occurrences', fontsize=18, labelpad=10)
plt.title('Keywords Popularity', bbox={'facecolor': 'k', 'pad': 5}, color='w', fontsize=25)
ax.bar(x_vals, y_vals, align='center', color='r')
plt.show()

In [9]:
missing_df = df_initial.isnull().sum(axis=0).reset_index()
missing_df.columns = ['column_name', 'missing_count']
missing_df['filling_factor'] = (df_initial.shape[0] - missing_df['missing_count']) / df_initial.shape[0] * 100
missing_df.sort_values('filling_factor').reset_index(drop=True)

The most popular approach in building a recommendation engine would be to use genres of similar movie category to present to the user based on the current movie.

In [10]:
genre_labels = set()
for s in df_initial['genres'].str.split('|').values:
    genre_labels = genre_labels.union(set(s))

In [11]:
genre_occurrences, _ = count_word(df_initial, 'genres', genre_labels)
genre_occurrences[:10]

In [12]:
fig = plt.figure(1, figsize=(18, 13))
ax = fig.add_subplot(2, 1, 1)

words_dict = {s[0]: s[1] for s in genre_occurrences[:25]}

wc = WordCloud(
    width=500, height=300,
    background_color='black',
    max_words=25,
    relative_scaling=1,
    normalize_plurals=False,
)
wc.generate_from_frequencies(words_dict)
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')

** Cleanup of data **


Clearly the plot_keywords is important feature of this dataset and there appears to be multiple synonyms that are quite similar to other keywords, for example: musicals and music, time traveller and time travel etc. These redundant keywords must be grouped under common roots. We collect all the keywords using the NLTK package.

**The Natural Language Toolkit, or more commonly NLTK**, is a suite of libraries and programs for symbolic and statistical natural language processing (NLP) for English written in the Python programming language. It was developed by Steven Bird and Edward Loper in the Department of Computer and Information Science at the University of Pennsylvania.NLTK includes graphical demonstrations and sample data. It is accompanied by a book that explains the underlying concepts behind the language processing tasks supported by the toolkit, plus a cookbook.

*Source: [Wikipedia](https://stackoverflow.com/questions/24647400/what-is-the-best-stemming-method-in-python)*

Here are some examples using the PorterStemmer
> import nltk
ps = nltk.stemmer.PorterStemmer()
ps.stem('grows')
'grow'
ps.stem('leaves')
'leav'
ps.stem('fairly')
'fairli'

*Source: [Wikipedia](https://en.wikipedia.org/wiki/Natural_Language_Toolkit)*


> **nltk.wordnet.WordNetLemmatizer() and nltk.stem.PorterStemmer():**
The goal of both stemming and lemmatization is to reduce inflectional forms and sometimes derivationally related forms of a word to a common base form.
However, the two words differ in their flavor. Stemming usually refers to a crude heuristic process that chops off the ends of words in the hope of achieving this goal correctly most of the time, and often includes the removal of derivational affixes. Lemmatization usually refers to doing things properly with the use of a vocabulary and morphological analysis of words, normally aiming to remove inflectional endings only and to return the base or dictionary form of a word, which is known as the lemma .


**From the NLTK docs:**

> Lemmatization and stemming are special cases of normalization. They identify a canonical representative for a set of related word forms.

In [13]:
df_cleaned = df_initial

In [14]:
def keywords_inventory(dataframe, colname='plot_keywords'):
    """Group keyword variants under common stems."""
    keywords_roots = {}
    keywords_select = {}
    category_keys = []
    for s in dataframe[colname]:
        if pd.isnull(s):
            continue
        for t in s.split('|'):
            t = t.lower()
            root = PS.stem(t)
            if root in keywords_roots:
                keywords_roots[root].add(t)
            else:
                keywords_roots[root] = {t}
    for s in keywords_roots:
        if len(keywords_roots[s]) > 1:
            leaf = min(keywords_roots[s], key=len)
            category_keys.append(leaf)
            keywords_select[s] = leaf
        else:
            val = list(keywords_roots[s])[0]
            category_keys.append(val)
            keywords_select[s] = val
    print(f"Number of keywords in variable '{colname}': {len(category_keys)}")
    return category_keys, keywords_roots, keywords_select

In [15]:
keywords, keywords_roots, keywords_select = keywords_inventory(df_cleaned, colname='plot_keywords')

In [16]:
i_count = 0
for s in keywords_roots:
    if len(keywords_roots[s]) > 1:
        i_count += 1
        if i_count < 15:
            print(i_count, keywords_roots[s], len(keywords_roots[s]))

In the next step of cleaning, suppress the keywords that appear less that 5 times and replace them by a synomym of higher frequency. As a second step, I suppress all the keywords that appear in less than 3 films

In [17]:
def replace_keywords(df, replacement_dict, roots=False):
    """Replace keyword variants with their canonical form."""
    df_new = df.copy(deep=True)
    for index, row in df_new.iterrows():
        chain = row['plot_keywords']
        if pd.isnull(chain):
            continue
        new_list = []
        for s in chain.split('|'):
            leaf = PS.stem(s) if roots else s
            if leaf in replacement_dict:
                new_list.append(replacement_dict[leaf])
            else:
                new_list.append(s)
        df_new.at[index, 'plot_keywords'] = '|'.join(new_list)
    return df_new

In [18]:
df_keywords_cleaned = replace_keywords(df_cleaned, keywords_select, roots=True)

In [19]:
keywords.remove('')
keyword_occurrences, keywords_count = count_word(df_keywords_cleaned, 'plot_keywords', keywords)
keyword_occurrences[:5]

In [20]:
def get_synonyms(word_parsed):
    """Get noun synonyms from WordNet."""
    lemma = set()
    for s in wordnet.synsets(word_parsed):
        for w in s.lemma_names():
            idx = s.name().find('.') + 1
            if s.name()[idx] == 'n':
                lemma.add(w.lower().replace('_', ' '))
    return lemma

In [21]:
word_parsed = 'alien'
lemma = get_synonyms(word_parsed)
for s in lemma:
    print(f' "{s:<30}" in keywords list -> {s in keywords} {keywords_count[s] if s in keywords else 0}')

In [22]:
def test_keyword(word_parsed, keycount, threshold):
    return keycount.get(word_parsed, 0) >= threshold

In [23]:
keyword_occurrences.sort(key=lambda x: x[1], reverse=False)
key_count = {s[0]: s[1] for s in keyword_occurrences}

replace_words = {}
i_count = 0
for index, (word_parsed, nb_apparitions) in enumerate(keyword_occurrences):
    if nb_apparitions > 5:
        continue
    lemma = get_synonyms(word_parsed)
    if len(lemma) == 0:
        continue
    candidates = [
        (s, key_count[s]) for s in lemma
        if test_keyword(s, key_count, key_count[word_parsed])
    ]
    candidates.sort(key=lambda x: (x[1], x[0]), reverse=True)
    if len(candidates) <= 1:
        continue
    if word_parsed == candidates[0][0]:
        continue
    i_count += 1
    if i_count < 8:
        print(f'{word_parsed:<12} -> {candidates[0][0]:<12} (init: {candidates})')
    replace_words[word_parsed] = candidates[0][0]

print('_' * 90)
print(f'The replacement concerns {round(len(replace_words) / len(keywords) * 100, 2)}% of the keywords.')

In [24]:
print('KEYWORDS THAT APPEAR BOTH IN KEYS AND VALUES:')
print('-' * 45)
i_count = 0
for s in replace_words.values():
    if s in replace_words:
        i_count += 1
        if i_count < 10:
            print(f'{s:<20} -> {replace_words[s]:<20}')

for key, value in replace_words.items():
    if value in replace_words:
        replace_words[key] = replace_words[value]

In [25]:
df_keywords_synonyms = replace_keywords(df_keywords_cleaned, replace_words, roots=False)
keywords, keywords_roots, keywords_select = keywords_inventory(df_keywords_synonyms, colname='plot_keywords')

In [26]:
keywords.remove('')
new_keyword_occurrences, keywords_count = count_word(df_keywords_synonyms, 'plot_keywords', keywords)
new_keyword_occurrences[:5]

Deleting keywords with low frequencies to improve prediction

In [27]:
def replace_low_frequency_keywords(df, keyword_occurrences_list):
    """Remove keywords that appear in fewer than 4 films."""
    df_new = df.copy(deep=True)
    key_count_dict = {s[0]: s[1] for s in keyword_occurrences_list}
    for index, row in df_new.iterrows():
        chain = row['plot_keywords']
        if pd.isnull(chain):
            continue
        new_list = [s for s in chain.split('|') if key_count_dict.get(s, 4) > 3]
        df_new.at[index, 'plot_keywords'] = '|'.join(new_list)
    return df_new

In [28]:
df_keywords_occurrence = replace_low_frequency_keywords(df_keywords_synonyms, new_keyword_occurrences)
keywords, keywords_roots, keywords_select = keywords_inventory(df_keywords_occurrence, colname='plot_keywords')

In [29]:
keywords.remove('')
new_keyword_occurrences, keywords_count = count_word(df_keywords_occurrence, 'plot_keywords', keywords)
new_keyword_occurrences[:5]

In [30]:
df_var_cleaned = df_keywords_occurrence.copy(deep=True)

In [31]:
missing_df = df_var_cleaned.isnull().sum(axis=0).reset_index()
missing_df.columns = ['column_name', 'missing_count']
missing_df['filling_factor'] = (df_var_cleaned.shape[0] - missing_df['missing_count']) / df_var_cleaned.shape[0] * 100
missing_df.sort_values('filling_factor').reset_index(drop=True)

In [32]:
df_filling = df_var_cleaned.copy(deep=True)
missing_year_info = df_filling[df_filling['title_year'].isnull()][[
    'director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name'
]]
missing_year_info[:10]

In [33]:
df_filling.iloc[4553]

In [34]:
df = df_filling.copy(deep=True)
missing_df = df.isnull().sum(axis=0).reset_index()
missing_df.columns = ['column_name', 'missing_count']
missing_df['filling_factor'] = (df.shape[0] - missing_df['missing_count']) / df.shape[0] * 100
missing_df.sort_values('filling_factor').reset_index(drop=True)

In [35]:
df = df_filling.copy(deep=True)
df.reset_index(inplace=True, drop=True)

** Recommendation engine**


In order to build the recommendation engine, we will basically proceed in two steps:

1. determine  N  films with a content similar to the entry provided by the user

2. select the 5 most popular films among these  N  films


**Similarity**

When builing the engine, the first step thus consists in defining a criteria that would tell us how close two films are. To do so, we start from the description of the film that was selected by the user: from it, we get the director name, the names of the actors and a few keywords. We then build a matrix where each row corresponds to a film of the database and where the columns correspond to the previous quantities (director + actors + keywords) plus the k genres.

In this matrix, the  aij  coefficients take either the value 0 or 1 depending on the correspondance between the significance of column  j  and the content of film  i . For exemple, if "keyword 1" is in film  i , we will have  aij  = 1 and 0 otherwise. Once this matrix has been defined, we determine the distance between two films according to:


\begin{eqnarray}
d_{m, n} = \sqrt{  \sum_{i = 1}^{N} \left( a_{m,i}  - a_{n,i} \right)^2  } 
\end{eqnarray}


**Popularity based**
According to similarities between entries, we get a list of  N  films. At this stage, we select 5 films from this list and, to do so, we give a score for every entry. we decide de compute the score according to 3 criteria:

* the IMDB score
* the number of votes the entry received
* the year of release


The two first criteria will be a direct measure of the popularity of the various entries in IMDB. For the third criterium, we introduce the release year since the database spans films from the early  XXth  century up to now. We assume that people's favorite films will be most of the time from the same epoch.

Then, we calculate the score according to the formula:

\begin{eqnarray}
\mathrm{score} = IMDB^2 \times \phi_{\sigma_1, c_1} \times  \phi_{\sigma_2, c_2}
\end{eqnarray}

where  ϕ  is a gaussian function:

\begin{eqnarray}
\phi_{\sigma, c}(x) \propto \mathrm{exp}\left(-\frac{(x-c)^2}{2 \, \sigma^2}\right)
\end{eqnarray}


For votes, we get the maximum number of votes among the  N  films and we set  σ1=c1=m . For years, I put  σ1=20  and I center the gaussian on the title year of the film selected by the user. With the gaussians, we put more weight to the entries with a large number of votes and to the films whose release year is close to the title selected by the user.

In [36]:
gaussian_filter = lambda x, y, sigma: math.exp(-(x - y) ** 2 / (2 * sigma ** 2))

The following function, **entryVariables**, returns values taken by variables 'director_name', 'actorN_name' wherre N => [1:3] and 'plot_keywords' for the film selected by the user.

In [37]:
def entry_variables(df, id_entry):
    """Extract director, actor, and keyword features for a given film."""
    col_labels = []
    if pd.notnull(df['director_name'].iloc[id_entry]):
        col_labels.extend(df['director_name'].iloc[id_entry].split('|'))

    for i in range(3):
        column = f'actor_{i + 1}_name'
        if pd.notnull(df[column].iloc[id_entry]):
            col_labels.extend(df[column].iloc[id_entry].split('|'))

    if pd.notnull(df['plot_keywords'].iloc[id_entry]):
        col_labels.extend(df['plot_keywords'].iloc[id_entry].split('|'))

    return col_labels

The following function, **addVariables** adds a list of variables to the dataframe given in input and initialize these variables at 0 or 1 depending on the correspondance with the description of the films and the content of the REF_VAR variable given in input.

In [38]:
def add_variables(df, ref_var):
    """Create binary columns for each feature in ref_var."""
    for s in ref_var:
        df[s] = 0
    col_names = ['genres', 'actor_1_name', 'actor_2_name',
                 'actor_3_name', 'director_name', 'plot_keywords']
    for col in col_names:
        for index, row in df.iterrows():
            if pd.isnull(row[col]):
                continue
            for s in row[col].split('|'):
                if s in ref_var:
                    df.at[index, s] = 1
    return df

The following function, **recommend**, creates a list of N (=31) films similar to the film selected by the user.

In [39]:
def recommend(df, id_entry):
    """Find the 31 most similar films using KNN on binary features."""
    df_copy = df.copy(deep=True)
    all_genres = set()
    for s in df['genres'].str.split('|').values:
        all_genres = all_genres.union(set(s))

    var = entry_variables(df_copy, id_entry)
    var += list(all_genres)
    df_new = add_variables(df_copy, var)

    x = df_new[var].to_numpy()
    nn = NearestNeighbors(n_neighbors=31, algorithm='auto', metric='euclidean').fit(x)
    x_test = df_new.iloc[id_entry][var].to_numpy().reshape(1, -1)
    distances, indices = nn.kneighbors(x_test)

    return indices[0][:]

The following function, **extractParameters** extracts some variables of the dataframe given in input and returns this list for a selection of N films. This list is ordered according to criteria established in the **critereSelection** function given below.

In [40]:
def extract_parameters(df, list_of_films):
    """Extract and rank film parameters by selection criteria."""
    para_list = ['_' for _ in range(31)]
    max_users = -1
    for i, index in enumerate(list_of_films):
        para_list[i] = list(df.iloc[index][[
            'movie_title', 'title_year', 'imdb_score',
            'num_user_for_reviews', 'num_voted_users'
        ]])
        para_list[i].append(index)
        max_users = max(max_users, para_list[i][4])

    main_title = para_list[0][0]
    year_ref = para_list[0][1]
    para_list.sort(
        key=lambda x: selection_criteria(main_title, max_users, year_ref, x[0], x[1], x[2], x[4]),
        reverse=True,
    )
    return para_list

The following function, **sequel**, compares the 2 titles passed in input and defines if these titles are similar or not.

In [41]:
def is_sequel(title1, title2):
    """Check if two titles are similar enough to be sequels."""
    return fuzz.ratio(title1, title2) > 50 or fuzz.token_set_ratio(title1, title2) > 50

The following function, **critereSelection **, compares the 2 titles passed in input and defines if these titles are similar or not.

In [42]:
def selection_criteria(main_title, max_users, year_ref, title, year, imdb_score, votes):
    """Score a candidate film based on release year proximity and vote count."""
    fact1 = gaussian_filter(year_ref, year, 20) if pd.notnull(year_ref) else 1
    sigma = max_users * 1.0
    fact2 = gaussian_filter(votes, max_users, sigma) if pd.notnull(votes) else 0

    if is_sequel(main_title, title):
        return 0
    return imdb_score ** 2 * fact1 * fact2

The following function, **addToSelection**, function complete the **filmSelect** list which contains 5 films that will be recommended to the user. The films are selected from the parametres_films list and are taken into account only if the title is different enough from other film titles.

In [43]:
def add_to_selection(film_select, para_list):
    """Add up to 5 non-duplicate films to the selection."""
    film_list = film_select[:]
    i_count = len(film_list)
    for i in range(31):
        is_in_list = any(
            s[0] == para_list[i][0] or is_sequel(para_list[i][0], s[0])
            for s in film_select
        )
        if is_in_list:
            continue
        i_count += 1
        if i_count <= 5:
            film_list.append(para_list[i])
    return film_list

The following function, **removeSequel** removes sequels from the list if more that two films from a serie are present. The older one is kept.

In [44]:
def remove_sequels(film_select):
    """Remove sequel duplicates, keeping the older film."""
    remove = []
    for i, film_1 in enumerate(film_select):
        for j, film_2 in enumerate(film_select):
            if j <= i:
                continue
            if is_sequel(film_1[0], film_2[0]):
                last_film = film_2[0] if film_1[1] < film_2[1] else film_1[0]
                remove.append(last_film)
    return [film for film in film_select if film[0] not in remove]

The following function is the **Main function** that will create a list of 5 films that will be recommended to the user.

In [45]:
def find_similar(df, id_entry, del_sequel=True, verbose=False):
    """Find 5 recommended films similar to the given entry."""
    if verbose:
        print('_' * 90)
        print(f"Query: Films similar to id = {id_entry}  -> '{df.iloc[id_entry]['movie_title']}'")

    list_recommend = recommend(df, id_entry)
    para_list = extract_parameters(df, list_recommend)

    film_select = add_to_selection([], para_list)
    if del_sequel:
        film_select = remove_sequels(film_select)
    film_select = add_to_selection(film_select, para_list)

    select_titles = []
    for i, s in enumerate(film_select):
        select_titles.append([s[0].replace('\xa0', ''), s[5]])
        if verbose:
            print(f"no{i + 1:<2}     -> {s[0]:<30}")

    return select_titles

**Making actual recommendations**

While building the recommendation engine, we are quickly faced to a big issue: the existence of sequels make that some recommendations may seem quite dumb. As an exemple, somebody who enjoyed "Pirates of the Caribbean: Dead Man's Chest" would probably not like to be adviced to watch this:

In [46]:
results = find_similar(df, 12, del_sequel=False, verbose=True)

The current engine is built in such a fashion that it is quite probable that if the engine matches one film of a series (like a trilogy), it will end recommending various of them. In the previous example, we see that the engine recommends the three films of the **Pirates of the Caribbean trilogy** majorly. 
Hence, we tried to find a way to prevent that kind of behaviour and I concluded that the quickest way to do it would be to work on the film's titles. To do so, I used the **fuzzywuzzy** package to build the **removeSequels** function. This function defines the degree of similarity of two film titles and if too close, the most recent film is removed from the list of recommendations. Using this function on the previous exemple, we end with the following recommendations:

In [47]:
results = find_similar(df, 12, del_sequel=True, verbose=True)

**Running a few test cases**

In [52]:
results = find_similar(df, 14, del_sequel=True, verbose=True)

In [51]:
results = find_similar(df, 14, del_sequel=False, verbose=True)

In [55]:
results = find_similar(df, 3, del_sequel=True, verbose=True)